## MLP

In [ ]:
import torch
import sys
import torch.nn as nn
import torch.optim as optim
sys.path.append("../")

from datasets.dataloader import get_wireless_dataloader

# Define Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, num_freqs=10, input_dim=3):
        super(PositionalEncoding, self).__init__()
        self.num_freqs = num_freqs
        self.output_dim = input_dim * (2 * num_freqs + 1)

    def forward(self, x):
        freqs = 2 ** torch.arange(self.num_freqs, dtype=torch.float32, device=x.device)
        x_expanded = x[..., None] * freqs  # Shape: [B, 3, num_freqs]
        x_encoded = torch.cat([x] + [torch.sin(x_expanded), torch.cos(x_expanded)], dim=-1)
        return x_encoded.view(x.shape[0], -1)  # Flatten

# Define MLP Model
class MLPChannelEstimator(nn.Module):
    def __init__(self, input_dim=3, num_freqs=10, hidden_dim=128, tx_ant=16, rx_ant=16):
        super(MLPChannelEstimator, self).__init__()
        self.tx_ant = tx_ant
        self.rx_ant = rx_ant
        self.pos_encoding = PositionalEncoding(num_freqs=num_freqs, input_dim=input_dim)
        encoded_dim = self.pos_encoding.output_dim

        # MLP with skip connections
        self.fc1 = nn.Linear(encoded_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim + encoded_dim, hidden_dim)  # Skip connection
        self.fc5 = nn.Linear(hidden_dim, hidden_dim)
        self.fc6 = nn.Linear(hidden_dim, hidden_dim)
        self.fc7 = nn.Linear(hidden_dim, hidden_dim)
        self.fc8 = nn.Linear(hidden_dim, tx_ant * rx_ant * 2)  # Output (real & imaginary parts)

    def forward(self, x):
        x = self.pos_encoding(x)  # Apply positional encoding
        x1 = torch.relu(self.fc1(x))
        x2 = torch.relu(self.fc2(x1))
        x3 = torch.relu(self.fc3(x2))
        x4 = torch.cat([x, x3], dim=-1)  # Skip connection
        x4 = torch.relu(self.fc4(x4))
        x5 = torch.relu(self.fc5(x4))
        x6 = torch.relu(self.fc6(x5))
        x7 = torch.relu(self.fc7(x6))
        x_out = self.fc8(x7)
        return x_out.view(self.tx_ant, self.rx_ant, 2)  # Output shape: (tx_ant, rx_ant, 2)

# Load Dataset
dataloader = get_wireless_dataloader(
    "../datasets/outputs/conf_16x2_414u_5.0ghz_sbrRT_sc104.mat",
    batch_size=16,
    num_pc=16378,
    drop_last=True
)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model, Loss, Optimizer
tx_ant, rx_ant = 16, 16
model = MLPChannelEstimator(tx_ant=tx_ant, rx_ant=rx_ant).to(device)
criterion = nn.MSELoss()  # Mean Squared Error for (I, Q) pairs
optimizer = optim.Adam(model.parameters(), lr=5e-4)

# Training Loop
num_epochs = 50  # Adjust as needed
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in dataloader:
        rx_positions = batch["rx_position"].to(device)  # [B, 3]
        channel_matrix = batch["channel_matrix"].to(device)  # [B, num_tx, num_rx, 2]

        # Forward pass
        predicted_channel = torch.stack([model(rx) for rx in rx_positions])  # [B, num_tx, num_rx, 2]

        # Compute loss
        loss = criterion(predicted_channel, channel_matrix)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}")

# Save Model
torch.save(model.state_dict(), "mlp_channel_estimator.pth")
print("Training Complete. Model saved.")
